# 🎨 ComfyUI + GGUF Image Generation Studio - ALL IN ONE
## Complete Workflow: ZImage Turbo Optimized for Free Tier

**Optimized for:** Google Colab Free Tier + Kaggle Free Tier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aa95-tech/colab-gguf-image-studio/blob/main/notebooks/05_ALL_IN_ONE_ZImageTurbo.ipynb)

---

## Quick Start

**Run cells in order:**
1. **Cell 1**: Environment Setup (Google Drive mount)
2. **Cell 2**: Install ComfyUI + GGUF
3. **Cell 3**: Download ZImage Turbo Model
4. **Cell 4**: Text-to-Image Generator
5. **Cell 5**: Image-to-Image with Character Reference
6. **Cell 6**: (Optional) ComfyUI Web Dashboard

---
## 🔧 CELL 1: Environment Setup & Google Drive Mount

In [ ]:
# CELL 1: Setup Environment & Mount Google Drive
import os
import sys
from pathlib import Path

# Detect environment
try:
    from google.colab import drive
    IS_COLAB = True
    print("✓ Google Colab detected")
except ImportError:
    IS_COLAB = False
    print("ℹ Local/Kaggle environment")

# Mount Google Drive (Colab only)
if IS_COLAB:
    drive.mount('/content/drive')
    MASTER_DIR = '/content/drive/MyDrive/ComfyUI_Studio'
elif os.path.exists('/kaggle/working'):
    MASTER_DIR = '/kaggle/working/ComfyUI_Studio'
else:
    MASTER_DIR = './ComfyUI_Studio'

# Create directory structure
os.makedirs(MASTER_DIR, exist_ok=True)
os.makedirs(f'{MASTER_DIR}/models', exist_ok=True)
os.makedirs(f'{MASTER_DIR}/models/checkpoints', exist_ok=True)
os.makedirs(f'{MASTER_DIR}/models/vae', exist_ok=True)
os.makedirs(f'{MASTER_DIR}/outputs', exist_ok=True)
os.makedirs(f'{MASTER_DIR}/ComfyUI', exist_ok=True)
os.makedirs(f'{MASTER_DIR}/reference_images', exist_ok=True)

print(f"\n✓ Master directory: {MASTER_DIR}")
print(f"✓ Models: {MASTER_DIR}/models")
print(f"✓ Outputs: {MASTER_DIR}/outputs")

# Store for later cells
%store MASTER_DIR
%store IS_COLAB

---
## 📦 CELL 2: Install ComfyUI + GGUF Support

In [ ]:
# CELL 2: Install ComfyUI & GGUF Support
import subprocess
import sys

%store -r MASTER_DIR

COMFYUI_DIR = f'{MASTER_DIR}/ComfyUI'

print("Installing ComfyUI + GGUF Framework...\n")

print("[1/5] Upgrading pip...")
subprocess.run(['pip', 'install', '--upgrade', 'pip'], capture_output=True)

print("[2/5] Installing PyTorch (CUDA optimized)...")
subprocess.run(['pip', 'install', 'torch', 'torchvision', 'torchaudio', 
               '--index-url', 'https://download.pytorch.org/whl/cu118'], capture_output=True)

print("[3/5] Cloning ComfyUI...")
if not os.path.exists(COMFYUI_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI.git', COMFYUI_DIR], 
                   capture_output=True)

print("[4/5] Installing ComfyUI dependencies...")
os.chdir(COMFYUI_DIR)
subprocess.run(['pip', 'install', '-r', 'requirements.txt'], capture_output=True)

print("[5/5] Installing GGUF & Diffusers support...")
subprocess.run(['pip', 'install', 'diffusers', 'transformers', 'llama-cpp-python', 'gguf', 'pillow', 'numpy'], 
               capture_output=True)

print("\n✓ ComfyUI + GGUF installed successfully!")
print(f"  Location: {COMFYUI_DIR}")

%store COMFYUI_DIR

---
## 🤖 CELL 3: Download ZImage Turbo Model

In [ ]:
# CELL 3: Download ZImage Turbo Model (Smart Caching)
import os
import subprocess

%store -r MASTER_DIR

MODELS_DIR = f'{MASTER_DIR}/models/checkpoints'
os.makedirs(MODELS_DIR, exist_ok=True)

# ZImage Turbo Model
MODEL_INFO = {
    'name': 'ZImage Turbo',
    'url': 'https://huggingface.co/lamm-mit/zimage-turbo/resolve/main/model.safetensors',
    'filename': 'zimage-turbo.safetensors',
    'size_gb': 2.5
}

model_path = f'{MODELS_DIR}/{MODEL_INFO["filename"]}'

print(f"Model: {MODEL_INFO['name']}")
print(f"Size: {MODEL_INFO['size_gb']}GB\n")

# Check if model already exists
if os.path.exists(model_path) and os.path.getsize(model_path) > 1024 * 1024:
    size_mb = os.path.getsize(model_path) / (1024 * 1024)
    print(f"✓ Model already downloaded ({size_mb:.0f}MB)")
else:
    print(f"Downloading {MODEL_INFO['name']} ({MODEL_INFO['size_gb']}GB)...")
    try:
        subprocess.run(['wget', '-q', '--show-progress', MODEL_INFO['url'], '-O', model_path], 
                       check=True, timeout=600)
        print(f"\n✓ Download complete!")
    except Exception as e:
        print(f"✗ Download failed: {str(e)}")
        print("Try again or download manually from:")
        print(MODEL_INFO['url'])

print(f"\n✓ Model ready: {model_path}")

%store MODELS_DIR
%store MODEL_INFO

---
## 🎨 CELL 4: Text-to-Image Generator (ZImage Turbo)

In [ ]:
# CELL 4: Text-to-Image Generation with ZImage Turbo
import torch
from diffusers import AutoPipelineForText2Image
from PIL import Image
import os
from datetime import datetime

%store -r MASTER_DIR
%store -r MODELS_DIR
%store -r MODEL_INFO
%store -r IS_COLAB

OUTPUT_DIR = f'{MASTER_DIR}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================
# 📝 EDIT YOUR PROMPT HERE
# ============================================
prompt = "a beautiful woman with long flowing hair, professional photography, studio lighting, 8k, cinematic"
negative_prompt = "blurry, low quality, distorted, bad anatomy"
num_inference_steps = 4  # ZImage Turbo is ultra-fast (4-20 steps recommended)
guidance_scale = 0.0  # ZImage Turbo works best with 0.0
height = 512
width = 512
num_images = 1
# ============================================

print(f"\n{'='*70}")
print("TEXT-TO-IMAGE GENERATION (ZImage Turbo)")
print(f"{'='*70}")
print(f"Prompt: {prompt[:60]}...")
print(f"Steps: {num_inference_steps} | Speed: ⚡ ULTRA-FAST")
print(f"Resolution: {width}x{height}")
print(f"\nGenerating {num_images} image(s)...\n")

try:
    # Setup device
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    
    if device == "cuda":
        print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    else:
        print(f"ℹ CPU Mode (slower)")
    
    print("[1/3] Loading ZImage Turbo pipeline...")
    model_path = f'{MODELS_DIR}/{MODEL_INFO["filename"]}'
    
    pipe = AutoPipelineForText2Image.from_single_file(
        model_path,
        torch_dtype=torch_dtype,
        safety_checker=None
    ).to(device)
    
    print("[2/3] Preparing inference...")
    generator = torch.Generator(device=device).manual_seed(42)
    
    print("[3/3] Generating images...")
    
    with torch.no_grad():
        images = pipe(
            prompt=[prompt] * num_images,
            negative_prompt=[negative_prompt] * num_images,
            height=height,
            width=width,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            generator=generator
        ).images
    
    print(f"\n✓ Generation complete!\n")
    
    # Save images
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    for idx, img in enumerate(images, 1):
        output_path = f"{OUTPUT_DIR}/txt2img_{timestamp}_{idx}.png"
        img.save(output_path)
        print(f"✓ Saved: txt2img_{timestamp}_{idx}.png")
        
        # Display in Colab
        if IS_COLAB:
            from IPython.display import display
            display(img)
    
    # Cleanup VRAM
    del pipe
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print(f"\n✓ All images saved to: {OUTPUT_DIR}")
    
except Exception as e:
    print(f"❌ Error: {str(e)}")
    import traceback
    traceback.print_exc()

---
## 📸 CELL 5: Image-to-Image with Character Reference

In [ ]:
# CELL 5: Image Upload & Character Analysis
import os
from pathlib import Path
from PIL import Image

%store -r MASTER_DIR
%store -r IS_COLAB

REFERENCE_DIR = f'{MASTER_DIR}/reference_images'
os.makedirs(REFERENCE_DIR, exist_ok=True)

print("\n" + "="*70)
print("IMAGE-TO-IMAGE: Upload Reference for Character Analysis")
print("="*70)

if IS_COLAB:
    from google.colab import files
    
    print("\nClick 'Choose Files' to upload a reference image...")
    print("Supported: .jpg, .png, .webp\n")
    
    uploaded = files.upload()
    
    uploaded_images = []
    for filename, data in uploaded.items():
        filepath = f"{REFERENCE_DIR}/{filename}"
        with open(filepath, 'wb') as f:
            f.write(data)
        uploaded_images.append(filepath)
        print(f"✓ Uploaded: {filename}")
else:
    print(f"Place images in: {REFERENCE_DIR}/")
    valid_extensions = {'.jpg', '.jpeg', '.png', '.webp'}
    uploaded_images = [f"{REFERENCE_DIR}/{f}" for f in os.listdir(REFERENCE_DIR) 
                       if Path(f).suffix.lower() in valid_extensions]
    
    if uploaded_images:
        print(f"Found {len(uploaded_images)} image(s)")
    else:
        print("No images found")

if uploaded_images:
    for img_path in uploaded_images:
        img = Image.open(img_path)
        print(f"\n📸 {os.path.basename(img_path)} | Size: {img.size}")
        if IS_COLAB:
            from IPython.display import display
            display(img.resize((300, 400)))

%store uploaded_images
%store REFERENCE_DIR

In [ ]:
# CELL 5b: Analyze Character Features with CLIP
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel

%store -r uploaded_images
%store -r IS_COLAB

if not uploaded_images:
    print("❌ No reference image uploaded. Run Cell 5 first.")
else:
    print("\n" + "="*70)
    print("CHARACTER FEATURE ANALYSIS")
    print("="*70)
    print("\n[1/3] Loading CLIP vision model...")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    print("[2/3] Analyzing image...")
    
    # Character analysis categories
    ANALYSIS_PROMPTS = {
        'Body Type': ['slim and lean', 'curvy and voluptuous', 'athletic', 'average', 'petite', 'tall and statuesque'],
        'Hair': ['long flowing', 'short', 'curly', 'straight', 'wavy', 'dark', 'blonde', 'red'],
        'Skin Tone': ['fair skin', 'olive skin', 'tan skin', 'dark skin'],
        'Facial Features': ['sharp features', 'soft features', 'high cheekbones', 'delicate', 'strong jawline'],
    }
    
    image = Image.open(uploaded_images[0]).convert('RGB')
    analysis_results = {}
    
    for category, prompts in ANALYSIS_PROMPTS.items():
        full_prompts = [f"A person with {p}" for p in prompts]
        
        with torch.no_grad():
            inputs = processor(text=full_prompts, images=[image]*len(prompts), 
                             return_tensors="pt", padding=True).to(device)
            outputs = model(**inputs)
            logits_per_image = outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1)[0].cpu().numpy()
        
        best_idx = probs.argmax()
        analysis_results[category] = {
            'detected': prompts[best_idx],
            'confidence': float(probs[best_idx])
        }
    
    print("\n[3/3] Analysis results:\n")
    print(f"{'CATEGORY':<20} | {'DETECTED':<25} | {'CONFIDENCE':<10}")
    print("-" * 70)
    
    for category, data in analysis_results.items():
        print(f"{category:<20} | {data['detected']:<25} | {data['confidence']:.1%}")
    
    # Generate character prompt
    character_prompt = ", ".join([v['detected'] for v in analysis_results.values()])
    
    print("\n" + "="*70)
    print("GENERATED CHARACTER PROMPT:")
    print("="*70)
    print(f"\n{character_prompt}")
    
    %store character_prompt
    %store analysis_results

In [ ]:
# CELL 5c: Image-to-Image Generation with Character Preservation
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
from datetime import datetime

%store -r MASTER_DIR
%store -r uploaded_images
%store -r character_prompt
%store -r IS_COLAB

OUTPUT_DIR = f'{MASTER_DIR}/outputs'

# ============================================
# 📝 EDIT YOUR IMAGE-TO-IMAGE PROMPT HERE
# ============================================
additional_prompt = "wearing a red dress, outdoor garden setting, professional photography, cinematic lighting, 8k"
negative_prompt = "blurry, distorted, low quality, bad anatomy"
strength = 0.75  # 0.0-1.0 (how much to modify the image)
guidance_scale = 7.5
num_inference_steps = 25
num_images = 1
# ============================================

if not uploaded_images:
    print("❌ No reference image. Run Cell 5 first.")
else:
    print(f"\n{'='*70}")
    print("IMAGE-TO-IMAGE GENERATION")
    print(f"{'='*70}")
    print(f"Reference: {os.path.basename(uploaded_images[0])}")
    print(f"Character: {character_prompt[:80]}...")
    print(f"New Context: {additional_prompt}")
    print(f"\nStrength: {strength} (0.5=subtle, 0.75=moderate, 1.0=maximum)\n")
    
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        
        print("[1/4] Loading Image-to-Image pipeline...")
        pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch_dtype,
            safety_checker=None
        ).to(device)
        
        print("[2/4] Loading reference image...")
        init_image = Image.open(uploaded_images[0]).convert('RGB').resize((512, 512))
        
        print("[3/4] Combining prompts...")
        full_prompt = f"{character_prompt}, {additional_prompt}"
        
        print("[4/4] Running generation...")
        
        with torch.no_grad():
            images = pipe(
                prompt=[full_prompt] * num_images,
                negative_prompt=[negative_prompt] * num_images,
                image=init_image,
                strength=strength,
                guidance_scale=guidance_scale,
                num_inference_steps=num_inference_steps
            ).images
        
        print(f"\n✓ Generation complete!\n")
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        for idx, img in enumerate(images, 1):
            output_path = f"{OUTPUT_DIR}/img2img_{timestamp}_{idx}.png"
            img.save(output_path)
            print(f"✓ Saved: img2img_{timestamp}_{idx}.png")
            
            if IS_COLAB:
                from IPython.display import display
                display(img)
        
        del pipe
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()

---
## 🌐 CELL 6: (Optional) ComfyUI Web Dashboard with Tunnel

In [ ]:
# CELL 6: ComfyUI Server + Public Tunnel (Optional)
import os
import subprocess
import time

%store -r MASTER_DIR
%store -r COMFYUI_DIR
%store -r IS_COLAB

COMFYUI_PORT = 8188

print("\n" + "="*70)
print("COMFYUI WEB DASHBOARD (Optional)")
print("="*70)

print("\n⚠️  This cell is OPTIONAL - for visual node-graph interface")
print("\nChoose tunnel type:")
print("  1. Cloudflare (recommended, most stable)")
print("  2. Ngrok (fast setup)")
print("  3. Local only (http://localhost:8188)\n")

TUNNEL_TYPE = "cloudflare"  # Change to 'ngrok' or 'local' as needed

print(f"[1/3] Preparing ComfyUI on port {COMFYUI_PORT}...")

# Link models
comfyui_models = f"{COMFYUI_DIR}/models"
master_models = f"{MASTER_DIR}/models"

if os.path.exists(comfyui_models) and os.path.islink(comfyui_models):
    os.unlink(comfyui_models)
elif os.path.exists(comfyui_models):
    import shutil
    shutil.rmtree(comfyui_models)

os.symlink(master_models, comfyui_models)
print("  ✓ Models linked")

print("[2/3] Starting ComfyUI server...")
os.chdir(COMFYUI_DIR)
subprocess.Popen(['python', 'main.py', '--port', str(COMFYUI_PORT)], 
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("  ⏳ Initializing (10 seconds)...")
time.sleep(10)

if TUNNEL_TYPE == "cloudflare":
    print("[3/3] Starting Cloudflare tunnel...")
    print("\n  Run in terminal:")
    print(f"  cloudflared tunnel --url http://localhost:{COMFYUI_PORT}")
    print("\n  Or use Ngrok:")
    print(f"  ngrok http {COMFYUI_PORT}")
elif TUNNEL_TYPE == "ngrok":
    print("[3/3] Setting up Ngrok tunnel...")
    subprocess.run(['pip', 'install', 'pyngrok'], capture_output=True)
    from pyngrok import ngrok
    public_url = ngrok.connect(COMFYUI_PORT)
    print(f"\n✓ Public URL: {public_url}")

print(f"\n✓ ComfyUI running on http://localhost:{COMFYUI_PORT}")
print("\nAccess the visual interface to:")
print("  • Build custom workflows")
print("  • Queue multiple jobs")
print("  • Monitor generation progress")
print("  • View all outputs")

---
## 📋 Usage Guide

### Complete Workflow Order:

1. **Cell 1**: Setup environment + Mount Google Drive
2. **Cell 2**: Install ComfyUI & dependencies (~3 minutes)
3. **Cell 3**: Download ZImage Turbo model (~2 minutes)
4. **Cell 4**: Generate images from text prompts ⚡
5. **Cell 5 (a-c)**: Upload reference → Analyze → Generate variations
6. **Cell 6** (Optional): Launch ComfyUI web interface

### Model Capabilities:

**ZImage Turbo:**
- ⚡ Ultra-fast (4-step generation)
- 🎨 High quality output
- 💾 Small model (2.5GB)
- ✅ Best for free tier

### Tips:

- **Prompts**: Be specific ("a woman, blue eyes, long hair" instead of "girl")
- **Steps**: 4-20 for ZImage Turbo (4 is fastest)
- **Strength**: 0.5=subtle changes, 0.75=moderate, 1.0=full regeneration
- **VRAM**: ZImage Turbo uses only 2-4GB (works on free GPU)

---

## 🎯 Common Issues & Fixes

**Problem**: "CUDA out of memory"
- **Solution**: Reduce `num_inference_steps` or image resolution

**Problem**: Model download fails
- **Solution**: Run Cell 3 again, or download from HuggingFace manually

**Problem**: Colab disconnected
- **Solution**: Models are saved in Google Drive - just reconnect and skip Cell 3

---

## 📚 Resources

- **GitHub Repo**: https://github.com/aa95-tech/colab-gguf-image-studio
- **ZImage Turbo**: https://huggingface.co/lamm-mit/zimage-turbo
- **Stable Diffusion**: https://huggingface.co/runwayml/stable-diffusion-v1-5
- **ComfyUI**: https://github.com/comfyanonymous/ComfyUI

---

**Happy Generating! 🎨✨**